# Stratified Schedule Audit

The experiment uses block randomization instead of a plain shuffle. Each
topic block has one A post and one B post, all posts use the same peak time,
and the schedule runs for 34 consecutive days.


In [ ]:
import pandas as pd
from experiment_analysis import load_config

config = load_config()
schedule = pd.read_csv(config.schedule_path)
schedule["scheduled_at"] = pd.to_datetime(schedule["scheduled_at"])
schedule["date"] = schedule["scheduled_at"].dt.date
schedule["time"] = schedule["scheduled_at"].dt.time.astype(str)
schedule["weekday"] = schedule["scheduled_at"].dt.day_name()

schedule.head()


In [ ]:
schedule["variant"].value_counts().sort_index()


In [ ]:
pd.crosstab(schedule["topic"], schedule["variant"])


In [ ]:
pd.crosstab(schedule["has_image"], schedule["variant"])


In [ ]:
pd.crosstab(schedule["weekday"], schedule["variant"]).reindex(
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
)


In [ ]:
assert len(schedule) == 34
assert schedule["date"].nunique() == 34
assert schedule["time"].nunique() == 1
assert schedule["time"].iloc[0] == config.posting_time
assert schedule["variant"].value_counts().to_dict() == {"A": 17, "B": 17}

topic_balance = pd.crosstab(schedule["topic"], schedule["variant"])
image_balance = pd.crosstab(schedule["has_image"], schedule["variant"])
weekday_balance = pd.crosstab(schedule["weekday"], schedule["variant"])

assert (topic_balance["A"] == topic_balance["B"]).all()
assert (image_balance["A"] == image_balance["B"]).all()
assert ((weekday_balance["A"] - weekday_balance["B"]).abs() <= 1).all()

print("Schedule passes stratified balance checks.")
